# 01 · Setup — the Environment API

This project has exactly one rule:

> **The attack agent never touches the world, the victim model, or the pixels.
> It only calls `reset()`, `observe()`, `step()`, `action_space()`,
> `observation_space()`.**

This notebook checks the install, loads the frozen victim, and takes a tour of
that API — including a demonstration that the "agent must not peek" rule is
enforced at runtime, not just documented.

In [ ]:
# Section 1: Setup
import sys, pathlib

ROOT = pathlib.Path.cwd()
if not (ROOT / "configs" / "default.yaml").exists():
    ROOT = ROOT.parent          # running from notebooks/
sys.path.insert(0, str(ROOT))

import numpy as np
import matplotlib.pyplot as plt

from configs.loader import load_config, build_victim, resolve

cfg = load_config()
print("project root:", ROOT)
print("victim config:", cfg["victim"])

In [ ]:
import torch, ultralytics, gymnasium, stable_baselines3
print("torch              ", torch.__version__)
print("ultralytics        ", ultralytics.__version__)
print("gymnasium          ", gymnasium.__version__)
print("stable-baselines3  ", stable_baselines3.__version__)

## Section 2: Environment

`BaseEnvironment` is the single interface all three stages implement.  Note that
the victim model, the renderer and the physics are *constructor arguments of the
environment* — they are never handed to an agent.

In [ ]:
from environments.base import BaseEnvironment
import inspect

print(inspect.getsource(BaseEnvironment)[:2000])

In [ ]:
from configs.objects import load_library

library = load_library(resolve(cfg["assets"]["objects_index"]), resolve(cfg["assets"]["objects_dir"]))
print(f"object library: {len(library)} objects")
for object_id in library.ids():
    a = library.get(object_id)
    print(f"  {a.id:<12} class={a.cls_name:<8} conf={a.confidence:.3f}  (from {a.source})")
print("\nStage 1 attacks a single one of these:", cfg["stage1"]["target_object"])

In [ ]:
victim = build_victim(cfg)          # frozen, pretrained, owned by the environment
print("victim:", victim.name)

from configs.loader import build_stage1_env
env = build_stage1_env(cfg, victim)   # a World: background + target cutout + attacker
clean = env.clean_image()
detections = env.victim_report(clean)   # experimenter-only call -- an agent could not make this
for d in sorted(detections, key=lambda d: -d.confidence)[:5]:
    print(f"  {d.cls_name:<12} {d.confidence:.3f}  {tuple(round(v) for v in d.bbox)}")

## Section 3: Visualization

In [ ]:
from PIL import Image
from evaluation.plots import draw_detections

target_asset = library.resolve(cfg["stage1"]["target_object"])
with Image.open(target_asset.path) as im:
    cutout = im.convert("RGBA")
checker = Image.new("RGBA", cutout.size, (235, 235, 235, 255))
checker.paste(cutout, (0, 0), cutout)

fig, axes = plt.subplots(1, 2, figsize=(11, 5.5))
axes[0].imshow(checker); axes[0].axis("off")
axes[0].set_title(f"{target_asset.id} -- a real cutout, own alpha, own object")
axes[1].imshow(draw_detections(clean, detections, highlight=cfg["victim"]["target_class"]))
axes[1].axis("off"); axes[1].set_title("Stage 1's clean World, rendered")
plt.show()

## Section 4: Baseline — the API tour

Everything an agent is allowed to know, printed out.

In [ ]:
from configs.loader import build_stage1_env

env = build_stage1_env(cfg, victim)
print("action_space():")
print(env.action_space().describe())
print()
print("observation_space():")
print(env.observation_space().describe())

obs = env.reset(seed=0)
print()
print("observation keys :", list(obs))
print("image shape      :", obs["image"].shape, obs["image"].dtype)
print("vector           :", np.round(obs["vector"], 3))

## Section 5: Attack Agent — and the wall between it and the world

`seal(env)` hands out a restricted handle.  The five public calls work; anything
else raises.  Every experiment in this project runs the agent through this
handle, so a rule violation cannot pass silently.

In [ ]:
from environments.sealed import seal, EnvironmentAccessError

api = seal(env)
obs = api.reset(seed=0)
obs, reward, terminated, truncated, info = api.step(api.action_space().sample(np.random.default_rng(0)))
print("step() ->", f"reward={reward:+.4f}", f"terminated={terminated}", f"truncated={truncated}", info)

for forbidden in ["_victim", "_world", "_baseline_conf", "render_human", "pop_telemetry"]:
    try:
        getattr(api, forbidden)
        print(f"{forbidden:<16} LEAKED (this would be a bug)")
    except EnvironmentAccessError as exc:
        print(f"{forbidden:<16} blocked -> {type(exc).__name__}")

## Section 6: Evaluation

The metric set is identical in all three stages, so results can be put in one
table: attack success rate, confidence drop, mean reward, movement cost,
episode length.

In [ ]:
from agents.random_agent import RandomAgent
from evaluation.runner import run_episodes
from evaluation.metrics import summarize
from evaluation.report import format_table

records, _ = run_episodes(env, RandomAgent(seed=0), n_episodes=10, method="random", seed=0)
summary = summarize(records)
summary["label"] = "random (smoke test)"
print(format_table([summary]))

## Section 7: Visualization

In [ ]:
plt.figure(figsize=(7, 3.6))
plt.plot([r.baseline_confidence for r in records], label="clean confidence")
plt.plot([r.best_confidence for r in records], label="after attack")
plt.xlabel("episode"); plt.ylabel("victim confidence"); plt.grid(alpha=.3); plt.legend()
plt.title("10 random placements through the Environment API")
plt.show()

Setup is complete when the cell above shows the attacked confidence dipping
below the clean line at least once.  Continue with `02_stage1_image.ipynb`.